In [158]:
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, classification_report, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score

In [159]:
df = pd.read_csv(
    "kagglehub/datasets/nalisha/job-salary-prediction-dataset/versions/1/job_salary_prediction_dataset.csv" #ignore the path its huge ik
    )

In [160]:
df.dtypes

job_title             str
experience_years    int64
education_level       str
skills_count        int64
industry              str
company_size          str
location              str
remote_work           str
certifications      int64
salary              int64
dtype: object

In [161]:
df["education_level"].unique()

<ArrowStringArray>
['Bachelor', 'PhD', 'High School', 'Diploma', 'Master']
Length: 5, dtype: str

In [162]:
df["company_size"].unique()

<ArrowStringArray>
['Medium', 'Small', 'Large', 'Enterprise', 'Startup']
Length: 5, dtype: str

# **Numerical Features**
- No need to clean, no null values
- No need to power transform, very little/insignificant skewdness
- Standardize via StandardScaler() and also see if results with RobustScaler() are better

# **Categorical Features**
- Ordinally encode education_level
- One-hot encode others

# **Feature selection**
- Feed principal components into SelectKBest, and select via ANOVA(f-regression) as it does not require the indices of categorical features.

# **Model**
- Test with different models - linear regression, random forrest, gradient boost regressor, etc...

In [163]:
transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [164]:
selector = SelectKBest(f_regression, k = 5)

In [165]:
pipeline = Pipeline([
    ("transform", transformations),
    #("selector", selector),
    ("model", LinearRegression())
])

In [166]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [167]:
models = [LinearRegression(), HistGradientBoostingRegressor()]

In [168]:
for model in models:
    pipeline.set_params(model = model)
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    print("-" * 50)
    print(f"R2: {r2_score(y_test, y_pred)}, MAE: {mean_absolute_error(y_test, y_pred)}")


--------------------------------------------------
R2: 0.9603931472072851, MAE: 5708.912556084143
--------------------------------------------------
R2: 0.9770681847906603, MAE: 4490.852225062369


In [169]:
print(pipeline.score(x_train, y_train))
print(pipeline.score(x_test, y_test))

0.977594343388526
0.9770681847906603


In [170]:
cross_val_score(
    pipeline,
    x,
    y,
    cv = 5,
    scoring = "r2"
)

array([0.97750565, 0.9773932 , 0.97742234, 0.97772371, 0.97756742])

In [171]:
import joblib
joblib.dump(pipeline, "salary_predictor.joblib")

['salary_predictor.joblib']